In [1]:
import sys
print(sys.executable)

c:\Users\sajin\Downloads\assistant-radiologue-virtuel-main\assistant-radiologue-virtuel-main\.venv\Scripts\python.exe


In [3]:
# === Cellule 1 : Premier test MedGemma sur 1 image ===
from pathlib import Path
import sys
import json
import time

# Ajoute la racine du projet au PYTHONPATH pour pouvoir importer src/
ROOT = Path('..').resolve()
sys.path.append(str(ROOT))

# Import du wrapper qu'on a écrit en étape 4
from src.medgemma_inference import medgemma_predict
from src.guardrails import apply_safety_guardrails

# Test sur UNE seule image (cas avec opacité simulée)
sample_image = ROOT / "data" / "sample_images" / "CXR_SYN_002_suspected_opacity.png"
print(f"📷 Image testée : {sample_image.name}")
print(f"⏱️ Lancement de l'inférence (le 1er appel télécharge ~4 Go de modèle)...\n")

t_start = time.perf_counter()
prediction = medgemma_predict(sample_image, mode="baseline")
prediction = apply_safety_guardrails(prediction)
t_end = time.perf_counter()

print(f"\n✅ Inférence terminée en {t_end - t_start:.1f} secondes")
print(f"\n📋 Résultat structuré :")
print(json.dumps(prediction, indent=2, ensure_ascii=False))

📷 Image testée : CXR_SYN_002_suspected_opacity.png
⏱️ Lancement de l'inférence (le 1er appel télécharge ~4 Go de modèle)...

⏳ Chargement de google/medgemma-4b-it (1ère fois = ~4 Go à télécharger)...


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

c:\Users\sajin\Downloads\assistant-radiologue-virtuel-main\assistant-radiologue-virtuel-main\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sajin\.cache\huggingface\hub\models--google--medgemma-4b-it. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

✅ MedGemma chargé en 181.2 s


[transformers] Deprecated: `processor.image_token` will switch from returning `tokenizer.image_token` to `tokenizer.boi_token` in v5.11.



✅ Inférence terminée en 328.2 secondes

📋 Résultat structuré :
{
  "image_quality": "good",
  "predicted_class": "uncertain",
  "confidence": 0.3,
  "visual_evidence": [
    "A circular opacity is present in the central chest region.",
    "The lung fields appear relatively clear, but the opacity is present."
  ],
  "justification": "The image shows a circular opacity in the central chest, which could be a normal anatomical structure or a sign of consolidation. The lung fields appear relatively clear, but the opacity is present. Further evaluation is needed to determine the cause of the opacity.",
  "limitations": [
    "The image is a single frontal view, which limits the assessment of the lung fields.",
    "The opacity's exact location and size are difficult to precisely determine."
  ],
  "warning": "Prototype pédagogique. Non destiné au diagnostic. Validation par un professionnel qualifié requise.",
  "model_name": "google/medgemma-4b-it",
  "prompt_version": "baseline_v1",
  "ra

In [4]:
# === Cellule 2 : Setup du loop complet ===
import time
import json
import pandas as pd
from tqdm import tqdm  # barre de progression

from src.guardrails import validate_prediction
from src.database import insert_run, init_db

# Chargement du dataset (30 cas)
cases_df = pd.read_csv(ROOT / "data" / "synthetic_cases.csv")
print(f"✅ {len(cases_df)} cas chargés")
print(f"Distribution : {dict(cases_df['label'].value_counts())}")

# Préparation SQLite (réutilise la base de S2/S3)
db_path = ROOT / "eval" / "medical_ai_evidence.sqlite"
init_db(db_path)

# Dossier de sortie
out_dir = ROOT / "eval" / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

print(f"✅ SQLite : {db_path.name}")
print(f"✅ Sortie : {out_dir}")
print(f"⏱️ Avec ~5 min par inférence, prévoir ~2h30 par mode, ~5h pour les 2 modes")

✅ 30 cas chargés
Distribution : {'normal': np.int64(10), 'suspected_opacity': np.int64(10), 'uncertain': np.int64(10)}
✅ SQLite : medical_ai_evidence.sqlite
✅ Sortie : C:\Users\sajin\Downloads\assistant-radiologue-virtuel-main\assistant-radiologue-virtuel-main\eval\outputs
⏱️ Avec ~5 min par inférence, prévoir ~2h30 par mode, ~5h pour les 2 modes


In [5]:
# === Cellule 3 : Sanity check sur 3 cas représentatifs ===
# Objectif : valider la boucle complète + saves incrémentales avant le run de 5h.

# On prend 1 cas par classe
sample_cases = cases_df.groupby("label").head(1).reset_index(drop=True)
print(f"🧪 Sanity check sur {len(sample_cases)} cas :")
print(sample_cases[["case_id", "label", "image_path"]])
print()

sanity_results = []
sanity_path = out_dir / "sanity_check_predictions.csv"

# Boucle avec barre de progression
for _, case in tqdm(sample_cases.iterrows(), total=len(sample_cases), desc="Sanity"):
    image_path = ROOT / case["image_path"]

    try:
        t_start = time.perf_counter()
        pred = apply_safety_guardrails(medgemma_predict(image_path, mode="baseline"))
        t_end = time.perf_counter()
        latency = int((t_end - t_start) * 1000)
        valid, errors = validate_prediction(pred)
        error_str = ""
    except Exception as e:
        # Si l'inférence plante : on log et on continue
        pred = {"predicted_class": "uncertain", "confidence": 0.0, "image_quality": "poor"}
        latency = -1
        valid = False
        error_str = str(e)[:200]

    insert_run(db_path, case["case_id"], str(image_path), pred)

    sanity_results.append({
        "case_id": case["case_id"],
        "label": case["label"],
        "predicted_class": pred["predicted_class"],
        "confidence": pred.get("confidence", 0),
        "image_quality": pred.get("image_quality", "?"),
        "latency_ms": latency,
        "latency_min": round(latency / 60000, 2),
        "json_valid": valid,
        "error": error_str,
    })

    # 🔒 SAUVEGARDE INCRÉMENTALE après chaque cas
    pd.DataFrame(sanity_results).to_csv(sanity_path, index=False, encoding="utf-8")

sanity_df = pd.DataFrame(sanity_results)
print(f"\n✅ Sanity check terminé")
print(f"   • Cas traités : {len(sanity_df)}")
print(f"   • Latence moyenne : {sanity_df['latency_min'].mean():.1f} min/image")
print(f"   • Latence totale : {sanity_df['latency_min'].sum():.1f} min")
print(f"   • Erreurs : {(sanity_df['error'] != '').sum()}")
print(f"   • CSV sauvé dans : {sanity_path.name}")

sanity_df

🧪 Sanity check sur 3 cas :
       case_id              label  \
0  CXR_SYN_001             normal   
1  CXR_SYN_002  suspected_opacity   
2  CXR_SYN_003          uncertain   

                                          image_path  
0          data/sample_images/CXR_SYN_001_normal.png  
1  data/sample_images/CXR_SYN_002_suspected_opaci...  
2       data/sample_images/CXR_SYN_003_uncertain.png  



Sanity: 100%|██████████| 3/3 [07:40<00:00, 153.62s/it]


✅ Sanity check terminé
   • Cas traités : 3
   • Latence moyenne : 2.6 min/image
   • Latence totale : 7.7 min
   • Erreurs : 0
   • CSV sauvé dans : sanity_check_predictions.csv


,case_id,label,predicted_class,confidence,image_quality,latency_ms,latency_min,json_valid,error
0,CXR_SYN_001,normal,uncertain,0.2,good,156711,2.61,True,
1,CXR_SYN_002,suspected_opacity,uncertain,0.3,good,147942,2.47,True,
2,CXR_SYN_003,uncertain,uncertain,0.2,limited,156150,2.60,True,


In [6]:
# === Cellule 4 : Run BASELINE complet (30 cas, ~1h15) ===
baseline_results = []
baseline_path = out_dir / "medgemma_baseline_predictions.csv"

print(f"🚀 Lancement du run baseline sur {len(cases_df)} cas...")
print(f"⏱️ ETA estimée : ~{len(cases_df) * 2.6:.0f} minutes ({len(cases_df) * 2.6 / 60:.1f} heures)")
print(f"💾 Sauvegarde incrémentale dans : {baseline_path.name}\n")

for _, case in tqdm(cases_df.iterrows(), total=len(cases_df), desc="Baseline"):
    image_path = ROOT / case["image_path"]

    try:
        t_start = time.perf_counter()
        pred = apply_safety_guardrails(medgemma_predict(image_path, mode="baseline"))
        t_end = time.perf_counter()
        latency = int((t_end - t_start) * 1000)
        valid, errors = validate_prediction(pred)
        error_str = ""
    except Exception as e:
        pred = {"predicted_class": "uncertain", "confidence": 0.0, "image_quality": "poor",
                "visual_evidence": [], "justification": "ERROR", "limitations": [str(e)[:200]]}
        latency = -1
        valid = False
        error_str = str(e)[:200]

    insert_run(db_path, case["case_id"], str(image_path), pred)

    baseline_results.append({
        "case_id": case["case_id"],
        "label": case["label"],
        "predicted_class": pred["predicted_class"],
        "confidence": pred.get("confidence", 0),
        "image_quality": pred.get("image_quality", "?"),
        "latency_ms": latency,
        "latency_min": round(latency / 60000, 2),
        "json_valid": valid,
        "error": error_str,
    })

    # 🔒 SAUVEGARDE INCRÉMENTALE après chaque cas
    pd.DataFrame(baseline_results).to_csv(baseline_path, index=False, encoding="utf-8")

baseline_df = pd.DataFrame(baseline_results)
print(f"\n✅ Run baseline terminé sur {len(baseline_df)} cas")
print(f"   • Latence totale : {baseline_df['latency_min'].sum():.1f} min")
print(f"   • Latence moyenne : {baseline_df['latency_min'].mean():.1f} min/image")
print(f"   • Erreurs : {(baseline_df['error'] != '').sum()}")
print(f"   • JSON valides : {baseline_df['json_valid'].sum()}/{len(baseline_df)}")
baseline_df.head(10)

🚀 Lancement du run baseline sur 30 cas...
⏱️ ETA estimée : ~78 minutes (1.3 heures)
💾 Sauvegarde incrémentale dans : medgemma_baseline_predictions.csv



Baseline: 100%|██████████| 30/30 [1:32:54<00:00, 185.80s/it]


✅ Run baseline terminé sur 30 cas
   • Latence totale : 92.9 min
   • Latence moyenne : 3.1 min/image
   • Erreurs : 0
   • JSON valides : 30/30


,case_id,label,predicted_class,confidence,image_quality,latency_ms,latency_min,json_valid,error
0,CXR_SYN_001,normal,uncertain,0.2,good,167403,2.79,True,
1,CXR_SYN_002,suspected_opacity,uncertain,0.3,good,164564,2.74,True,
2,CXR_SYN_003,uncertain,uncertain,0.2,limited,192016,3.20,True,
3,CXR_SYN_004,normal,normal,0.8,good,182001,3.03,True,
4,CXR_SYN_005,suspected_opacity,uncertain,0.3,good,181152,3.02,True,
5,CXR_SYN_006,uncertain,uncertain,0.2,limited,172449,2.87,True,
6,CXR_SYN_007,normal,uncertain,0.2,good,173931,2.90,True,
7,CXR_SYN_008,suspected_opacity,uncertain,0.3,good,180570,3.01,True,
8,CXR_SYN_009,uncertain,uncertain,0.2,limited,191610,3.19,True,
9,CXR_SYN_010,normal,uncertain,0.2,good,181575,3.03,True,


In [6]:
# === Cellule 5 : Métriques MedGemma baseline (30 synthétiques) ===
import sys
import json
from pathlib import Path
sys.path.append(str(Path('..').resolve()))

from src.metrics import accuracy, macro_f1

y_true = baseline_df["label"].tolist()
y_pred = baseline_df["predicted_class"].tolist()
valid_classes = {"normal", "suspected_opacity", "uncertain"}
hallucinations = baseline_df[~baseline_df["predicted_class"].isin(valid_classes)]

metrics = {
    "mode": "medgemma_baseline_synthetic",
    "n_cases": len(baseline_df),
    "accuracy": round(accuracy(y_true, y_pred), 4),
    "macro_f1": round(macro_f1(y_true, y_pred), 4),
    "json_valid_rate": round(baseline_df["json_valid"].mean(), 4),
    "uncertain_rate": round((baseline_df["predicted_class"] == "uncertain").mean(), 4),
    "latency_min_median": round(baseline_df["latency_ms"].median() / 60000, 2),
    "hallucination_count": len(hallucinations),
}

print("Metriques MedGemma baseline (30 synthetiques) :")
print(json.dumps(metrics, indent=2))

print("\nVerification des criteres du brief :")
checks = [
    ("Accuracy >= 0.70", metrics["accuracy"] >= 0.70, metrics["accuracy"]),
    ("Macro-F1 >= 0.68", metrics["macro_f1"] >= 0.68, metrics["macro_f1"]),
    ("JSON valides >= 95%", metrics["json_valid_rate"] >= 0.95, metrics["json_valid_rate"]),
    ("Hallucinations = 0", metrics["hallucination_count"] == 0, metrics["hallucination_count"]),
]
for label, ok, val in checks:
    icon = "OK" if ok else "KO"
    print(f"  [{icon}] {label} : {val}")

metrics_path = out_dir / "medgemma_baseline_synthetic_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(f"\nSauve dans : {metrics_path.name}")

NameError: name 'baseline_df' is not defined